# SQL

## CELL 1: Imports + DB connection 

In [48]:
import sqlite3
import pandas as pd
import re

conn = sqlite3.connect("../data/wc_analytics.db")
print("Connected to wc_analytics.db")

Connected to wc_analytics.db


## CELL 2: Load raw CSVs into SQLite tables 

In [49]:
results      = pd.read_csv("../data/raw/results.csv",                          encoding="latin-1")
wc_finals    = pd.read_csv("../data/raw/List of FIFA World Cup finals.csv",    encoding="latin-1")
wc_attendance= pd.read_csv("../data/raw/FIFA World Cup Attendance.csv",        encoding="latin-1")
wc_awards    = pd.read_csv("../data/raw/FIFA World Cup Award.csv",             encoding="latin-1")
wc_top4      = pd.read_csv("../data/raw/Teams reaching the top four.csv",      encoding="latin-1")
fifa_ranking = pd.read_csv("../data/raw/fifa_ranking-2023-07-20.csv",          encoding="latin-1")
results.to_sql("results",       conn, index=False, if_exists="replace")
wc_finals.to_sql("wc_finals",   conn, index=False, if_exists="replace")
wc_attendance.to_sql("wc_attendance", conn, index=False, if_exists="replace")
wc_awards.to_sql("wc_awards",   conn, index=False, if_exists="replace")
wc_top4.to_sql("wc_top4",       conn, index=False, if_exists="replace")
fifa_ranking.to_sql("rankings", conn, index=False, if_exists="replace")
 
print("All 6 tables loaded into SQLite database successfully.")

All 6 tables loaded into SQLite database successfully.


#  SECTION A — CLEANING: results.csv 
###  CELL 3: results — drop unused columns 
#### city: too granular, country is enough for host analysis
#### id: just a row number, pandas handles this automatically

In [50]:
# city: too granular, country is enough for host analysis
# id: just a row number, pandas handles this automatically
results_step1 = pd.read_sql_query("""
    SELECT
        date,
        home_team,
        away_team,
        home_score,
        away_score,
        tournament,
        country
    FROM results
""", conn)
print(f"Before: columns = {results.shape[1]} -> After: columns = {results_step1.shape[1]}")
print("Dropped: city, id")
print(results_step1.head(3))

Before: columns = 9 -> After: columns = 7
Dropped: city, id
         date home_team away_team  home_score  away_score tournament   country
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Scotland
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   England
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Scotland


##  CELL 4: results — fix Copa América encoding
#### The character é is stored as 'Ã©' due to latin-1 encoding
##### This fixes it so the tournament name is correct

In [51]:
results_step2 = pd.read_sql_query("""
    SELECT
        date,
        home_team,
        away_team,
        home_score,
        away_score,
        REPLACE(tournament, 'Copa AmÃ©rica', 'Copa América') AS tournament,
        country
    FROM results
""", conn)
 
# Verify fix worked
copa_count = results_step2[results_step2["tournament"] == "Copa América"].shape[0]
print(f"Copa América matches after fix: {copa_count}")

Copa América matches after fix: 841


## CELL 5: results — add result column (W/D/L from home perspective)
### H = home team won, A = away team won, D = draw
##### This makes win/loss analysis much easier later in EDA

In [52]:
results_step3 = pd.read_sql_query("""
    SELECT
        date,
        home_team,
        away_team,
        home_score,
        away_score,
        REPLACE(tournament, 'Copa AmÃ©rica', 'Copa América') AS tournament,
        country,
        CASE
            WHEN home_score > away_score THEN 'H'
            WHEN home_score < away_score THEN 'A'
            ELSE 'D'
        END AS result
    FROM results
    WHERE home_score IS NOT NULL
      AND away_score IS NOT NULL
""", conn)
 
print("Result column value counts:")
print(results_step3["result"].value_counts())
print(f"\nTotal matches: {len(results_step3):,}")

Result column value counts:
result
H    21031
A    12187
D    10059
Name: count, dtype: int64

Total matches: 43,277


### CELL 6: results — convert date to datetime 

In [53]:
results_clean = results_step3.copy()
results_clean["date"] = pd.to_datetime(results_clean["date"])
 
print(f"Date range: {results_clean['date'].min().date()} → {results_clean['date'].max().date()}")
print(f"Shape: {results_clean.shape}")
print("\nFinal columns:", results_clean.columns.tolist())
 
 
# ── CELL 7: results — quick sanity check ─────────────────────
print("=== RESULTS CLEANING — FINAL CHECKS ===\n")
print("Null values:")
print(results_clean.isnull().sum())
print()
print("Key tournament counts:")
key = ["FIFA World Cup", "Copa América", "UEFA Euro",
       "FIFA World Cup qualification", "UEFA Euro qualification"]
print(results_clean[results_clean["tournament"].isin(key)]["tournament"].value_counts())

Date range: 1872-11-30 → 2023-11-23
Shape: (43277, 8)

Final columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'country', 'result']
=== RESULTS CLEANING — FINAL CHECKS ===

Null values:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
country       0
result        0
dtype: int64

Key tournament counts:
tournament
FIFA World Cup qualification    8012
UEFA Euro qualification         2815
FIFA World Cup                   964
Copa América                     841
UEFA Euro                        337
Name: count, dtype: int64


### CELL 7: results — quick sanity check 


In [54]:
wc_finals_clean = pd.read_sql_query("""
    SELECT
        Year,
        Host,
        Champion,
        Runner_up,
        Third,
        Fourth,
        "No. _ofteams" AS num_teams
    FROM wc_finals
    WHERE Champion != '(Not held because of World War II)'
      AND Champion IS NOT NULL
    ORDER BY Year
""", conn)
 
print(f"WC Finals — {len(wc_finals_clean)} tournaments (WW2 years removed)")
print(wc_finals_clean.to_string(index=False))

WC Finals — 22 tournaments (WW2 years removed)
 Year               Host     Champion      Runner_up         Third       Fourth num_teams
 1930            Uruguay      Uruguay      Argentina United States   Yugoslavia        13
 1934              Italy        Italy Czechoslovakia       Germany      Austria        16
 1938             France        Italy        Hungary        Brazil       Sweden        15
 1950             Brazil      Uruguay         Brazil        Sweden        Spain        13
 1954        Switzerland West Germany        Hungary       Austria      Uruguay        16
 1958             Sweden       Brazil         Sweden        France West Germany        16
 1962              Chile       Brazil Czechoslovakia         Chile   Yugoslavia        16
 1966            England      England   West Germany      Portugal Soviet Union        16
 1970             Mexico       Brazil          Italy  West Germany      Uruguay        16
 1974       West Germany West Germany    Netherlands 

#  SECTION B — CLEANING: List of FIFA World Cup finals.csv
### CELL 8: wc_finals — removing bad rows + unused columns 
##### Score and Score.1 have venue names embedded inside them
##### e.g. "4–2 Estadio Centenario, Montevideo" — not usable
##### Also removes 1942 and 1946 rows (WW2, tournaments not held)

In [55]:
wc_finals_clean["Year"] = pd.to_numeric(wc_finals_clean["Year"], errors="coerce")
wc_finals_clean = wc_finals_clean.dropna(subset=["Year"])
wc_finals_clean["Year"] = wc_finals_clean["Year"].astype(int)
 
print(f"Years covered: {wc_finals_clean['Year'].min()} → {wc_finals_clean['Year'].max()}")
print(f"Shape: {wc_finals_clean.shape}")

Years covered: 1930 → 2022
Shape: (22, 7)


### CELL 9: wc_finals — convert Year to integer

In [56]:
wc_finals_clean["Year"] = pd.to_numeric(wc_finals_clean["Year"], errors="coerce")
wc_finals_clean = wc_finals_clean.dropna(subset=["Year"])
wc_finals_clean["Year"] = wc_finals_clean["Year"].astype(int)
 
print(f"Years covered: {wc_finals_clean['Year'].min()} → {wc_finals_clean['Year'].max()}")
print(f"Shape: {wc_finals_clean.shape}")

Years covered: 1930 → 2022
Shape: (22, 7)


#  SECTION C — CLEANING: Teams reaching the top four.csv
### CELL 10: top 4 — fixing 'Germany1' naming error

In [57]:
wc_top4_step1 = pd.read_sql_query("""
    SELECT
        REPLACE(Team, 'Germany1', 'Germany') AS team,
        Titles,
        "Runners-up"    AS runners_up,
        "Third place"   AS third_place,
        "Fourth place"  AS fourth_place,
        "Top 4 Total"   AS top4_total
    FROM wc_top4
    WHERE Team IS NOT NULL
      AND Team != 'Unnamed: 0'
""", conn)
 
print("Teams after Germany fix:")
print(wc_top4_step1["team"].tolist())

Teams after Germany fix:
['Brazil', 'Germany', 'Italy', 'Argentina', 'France', 'Uruguay', 'England', 'Spain', 'Netherlands', 'Hungary', 'Czech Republic2', 'Sweden', 'Croatia', 'Poland', 'Austria', 'Portugal', 'Belgium', 'United States', 'Chile', 'Turkey', 'Serbia3', 'Russia4', 'Bulgaria', 'South Korea', 'Morocco']


# CELL 11: top4 — extracting numbers from text columns
### Titles column looks like: "5 (1958, 1962, 1970, 1994, 2002)"
##### We only need the number — extract it with regex

In [58]:
def extract_number(text):
    """Pull the first number out of strings like '5 (1958, 1962...)'"""
    if pd.isna(text) or text == "NaN":
        return 0
    match = re.match(r"(\d+)", str(text).strip())
    return int(match.group(1)) if match else 0
 
wc_top4_clean = wc_top4_step1.copy()
for col in ["Titles", "runners_up", "third_place", "fourth_place"]:
    new_col = col.lower().replace(" ", "_") + "_count" if col == "Titles" else col + "_count"
    wc_top4_clean[new_col] = wc_top4_clean[col].apply(extract_number)
 
# Keep only clean numeric columns
wc_top4_clean = wc_top4_clean[[
    "team", "titles_count", "runners_up_count",
    "third_place_count", "fourth_place_count", "top4_total"
]].rename(columns={"Titles_count": "titles"})
 
print(wc_top4_clean.to_string(index=False))

           team  titles_count  runners_up_count  third_place_count  fourth_place_count  top4_total
         Brazil             5                 2                  2                   2          11
        Germany             4                 4                  4                   1          13
          Italy             4                 2                  1                   1           8
      Argentina             3                 3                  0                   0           6
         France             2                 2                  2                   1           7
        Uruguay             2                 0                  0                   3           5
        England             1                 0                  0                   2           3
          Spain             1                 0                  0                   1           2
    Netherlands             0                 3                  1                   1           5
        Hu

#  SECTION D — CLEANING: FIFA World Cup Attendance.csv
### CELL 12: attendance drop granular columns

In [59]:
wc_attendance_clean = pd.read_sql_query("""
    SELECT
        Year,
        Hosts,
        Total_Attendance,
        Matches,
        Average_Attendance
    FROM wc_attendance
    WHERE Year IS NOT NULL
    ORDER BY Year
""", conn)
 
wc_attendance_clean["Year"] = pd.to_numeric(wc_attendance_clean["Year"], errors="coerce")
wc_attendance_clean = wc_attendance_clean.dropna(subset=["Year"])
wc_attendance_clean["Year"] = wc_attendance_clean["Year"].astype(int)
 
print(f"Attendance data: {len(wc_attendance_clean)} tournaments")
print(wc_attendance_clean.tail(5).to_string(index=False))

Attendance data: 22 tournaments
 Year        Hosts  Total_Attendance  Matches  Average_Attendance
 2006      Germany           3359439       64               52491
 2010 South Africa           3178856       64               49670
 2014       Brazil           3429873       64               53592
 2018       Russia           3031768       64               47371
 2022        Qatar           3404252       64               53191


#  SECTION E — CLEANING: FIFA World Cup Award.csv
###  CELL 13: awards — fix column names (they have spaces) 

In [60]:

wc_awards_clean = pd.read_sql_query("""
    SELECT
        "World _Cup"    AS world_cup,
        "Golden _Ball"  AS golden_ball,
        "Golden _Boot"  AS golden_boot,
        Goals           AS golden_boot_goals,
        "Golden _Glove" AS golden_glove
    FROM wc_awards
    WHERE "World _Cup" IS NOT NULL
""", conn)
 
# Fix encoding on player names (accented characters)
for col in ["golden_ball", "golden_boot", "golden_glove"]:
    wc_awards_clean[col] = wc_awards_clean[col].apply(
        lambda x: x.encode("latin-1").decode("utf-8", errors="ignore")
        if isinstance(x, str) else x
    )
 
print(f"Awards data: {len(wc_awards_clean)} tournaments")
print(wc_awards_clean.tail(8).to_string(index=False))

Awards data: 22 tournaments
             world_cup     golden_ball                   golden_boot  golden_boot_goals       golden_glove
    1994 United States         Romário Oleg Salenko Hristo Stoichkov                  6 Michel Preud'homme
           1998 France         Ronaldo                   Davor Šuker                  6     Fabien Barthez
2002 South Korea/Japan     Oliver Kahn                       Ronaldo                  8        Oliver Kahn
          2006 Germany Zinedine Zidane                Miroslav Klose                  5   Gianluigi Buffon
     2010 South Africa    Diego Forlán                 Thomas Müller                  5      Iker Casillas
           2014 Brazil    Lionel Messi               James Rodríguez                  6       Manuel Neuer
           2018 Russia     Luka Modrić                    Harry Kane                  6   Thibaut Courtois
            2022 Qatar    Lionel Messi                 Kylian Mbappé                  8  Emiliano Martínez


#  SECTION F: CLEANING - fifa_ranking.csv
### CELL 14: rankings — drop uninformative columns
#### previous_points: redundant, we can derive this ourselves
##### rank_change: counterintuitive direction, we'll build our own

In [61]:

rankings_step1 = pd.read_sql_query("""
    SELECT
        rank,
        country_full,
        country_abrv,
        total_points,
        confederation,
        rank_date
    FROM rankings
    ORDER BY rank_date, rank
""", conn)
 
print(f"Rankings shape: {rankings_step1.shape}")
print("Columns kept:", rankings_step1.columns.tolist())

Rankings shape: (64757, 6)
Columns kept: ['rank', 'country_full', 'country_abrv', 'total_points', 'confederation', 'rank_date']


# CELL 15: rankings — standardise team names 
### These names differ between rankings and results.csv
##### If not fixed, joins between the two datasets will silently fail

In [62]:
# ── CELL 15: rankings — standardise team names ────────────────
# These names differ between rankings and results.csv
# If not fixed, joins between the two datasets will silently fail
name_fixes = {
    "TÃ¼rkiye"               : "Turkiye",
    "Turkey"                 : "Turkiye",
    "CÃ´te d'Ivoire"         : "Cote d'Ivoire",
    "Ivory Coast"            : "Cote d'Ivoire",
    "CuraÃ§ao"               : "Curacao",
    "SÃ£o TomÃ© e PrÃ­ncipe": "Sao Tome and Principe",
    "SÃ£o TomÃ© and PrÃ­ncipe": "Sao Tome and Principe",
    "Cape Verde Islands"     : "Cape Verde",
    "St. Vincent / Grenadines": "St. Vincent and the Grenadines",
    "FYR Macedonia"          : "North Macedonia",
}
 
rankings_step1["country_full"] = rankings_step1["country_full"].replace(name_fixes)
rankings_step1["rank_date"]    = pd.to_datetime(rankings_step1["rank_date"])
 
print("Name fixes applied ✓")
print(f"Date range: {rankings_step1['rank_date'].min().date()} → {rankings_step1['rank_date'].max().date()}")
print(f"Unique teams: {rankings_step1['country_full'].nunique()}")
 
rankings_clean = rankings_step1.copy()

Name fixes applied ✓
Date range: 1992-12-31 → 2023-07-20
Unique teams: 226


# CELL 16: rankings extract pre-WC snapshots
### For the ML model we need one ranking row per team per WC year
#### This gets the last available ranking before each tournament

In [63]:
wc_years = [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]
snapshots = []
 
for yr in wc_years:
    # Get all ranking dates in the 3 months before the tournament
    pre_wc = rankings_clean[
        (rankings_clean["rank_date"].dt.year == yr) &
        (rankings_clean["rank_date"].dt.month.isin([3, 4, 5, 6]))
    ]
    if len(pre_wc) > 0:
        # Take the most recent one
        latest_date = pre_wc["rank_date"].max()
        snapshot = pre_wc[pre_wc["rank_date"] == latest_date].copy()
        snapshot["wc_year"] = yr
        snapshots.append(snapshot)
        print(f"WC {yr}: using rankings from {latest_date.date()} ({len(snapshot)} teams)")
 
pre_wc_rankings = pd.concat(snapshots, ignore_index=True)
print(f"\nPre-WC rankings table shape: {pre_wc_rankings.shape}")

WC 1994: using rankings from 1994-06-14 (159 teams)
WC 1998: using rankings from 1998-05-20 (187 teams)
WC 2002: using rankings from 2002-05-15 (202 teams)
WC 2006: using rankings from 2006-05-17 (204 teams)
WC 2010: using rankings from 2010-05-26 (201 teams)
WC 2014: using rankings from 2014-06-05 (206 teams)
WC 2018: using rankings from 2018-06-07 (205 teams)
WC 2022: using rankings from 2022-06-23 (211 teams)

Pre-WC rankings table shape: (1575, 7)


#  SECTION G — Saving All Cleaned Files
### CELL 17: Save everything to data/cleaned/

In [64]:
import os
os.makedirs("../data/cleaned", exist_ok=True)
 
results_clean.to_csv("../data/cleaned/results_clean.csv",            index=False)
wc_finals_clean.to_csv("../data/cleaned/wc_finals_clean.csv",        index=False)
wc_top4_clean.to_csv("../data/cleaned/wc_top4_clean.csv",            index=False)
wc_attendance_clean.to_csv("../data/cleaned/wc_attendance_clean.csv",index=False)
wc_awards_clean.to_csv("../data/cleaned/wc_awards_clean.csv",        index=False)
rankings_clean.to_csv("../data/cleaned/rankings_clean.csv",          index=False)
pre_wc_rankings.to_csv("../data/cleaned/pre_wc_rankings.csv",        index=False)
 
print("All cleaned files saved to data/cleaned/ ✓\n")
 
cleaned_files = {
    "results_clean.csv"       : results_clean,
    "wc_finals_clean.csv"     : wc_finals_clean,
    "wc_top4_clean.csv"       : wc_top4_clean,
    "wc_attendance_clean.csv" : wc_attendance_clean,
    "wc_awards_clean.csv"     : wc_awards_clean,
    "rankings_clean.csv"      : rankings_clean,
    "pre_wc_rankings.csv"     : pre_wc_rankings,
}
 
print(f"{'File':<35} {'Rows':>8} {'Columns':>10}")
print("-" * 56)
for fname, df in cleaned_files.items():
    print(f"{fname:<35} {df.shape[0]:>8,} {df.shape[1]:>10}")

All cleaned files saved to data/cleaned/ ✓

File                                    Rows    Columns
--------------------------------------------------------
results_clean.csv                     43,277          8
wc_finals_clean.csv                       22          7
wc_top4_clean.csv                         25          6
wc_attendance_clean.csv                   22          5
wc_awards_clean.csv                       22          5
rankings_clean.csv                    64,757          6
pre_wc_rankings.csv                    1,575          7


 ## CLEANING COMPLETE
#### Fixed in results_clean.csv:
  ✓ Dropped city, id
  ✓ Fixed Copa América encoding
  ✓ Added result column (H / A / D)
  ✓ Removed rows with null scores
  ✓ Date converted to datetime
 
#### Fixed in wc_finals_clean.csv:
  ✓ Removed WW2 placeholder rows (1942, 1946)
  ✓ Dropped messy Score columns
  ✓ Renamed No._ofteams → num_teams
 
#### Fixed in wc_top4_clean.csv:
  ✓ Germany1 → Germany
  ✓ Extracted numbers from text columns
  ✓ Clean integer columns for ML features
 
#### Fixed in rankings_clean.csv:
  ✓ Dropped previous_points, rank_change
  ✓ Standardised 8 team name mismatches
  ✓ Date converted to datetime
 
#### New file: pre_wc_rankings.csv
  ✓ One ranking snapshot per team per WC year
  ✓ Ready to use directly as ML features
 
## Next → Notebook 03: EDA and Historical Analysis